# film

> A demo as a narrated explainer video, drawn with manimgl.

A film tells the story of one demo. It asks the gate question, counts down so that the viewer can predict, moves the input, and shows the answer.
The demo loop stays the same: predict, reveal, manipulate, explain.

A film has two parts:

1. `plan` turns a spec into film data. Python computes every value with the twins of the fns, as `run` does. The plan holds only numbers and text.
2. `render` gives the plan to a fixed manimgl scene, `rupaka/manim/film_scene.py`. The scene draws the data. It runs no code from the spec.

So a model can write the spec, and the film is still safe. The voice is not here: a host such as abhyasa speaks each line and gives the seconds to `timed`.

In [ ]:
#| default_exp film

In [ ]:
#| hide
from fastcore.test import *
from rupaka.spec import gallery

In [ ]:
#| export
import re, io, os, json, wave, shutil, tempfile, subprocess
from copy import deepcopy
from pathlib import Path
from fastcore.utils import *
from rupaka.spec import PKG, run, fmt_text, get, show_value, _gates, _walk

## Colours and timing

A film has two palettes. `board` is a dark board. `paper` is the warm ground of the paper theme.

In [ ]:
#| export
PALETTES = dict(
    board=dict(bg='#15171c', fg='#ece8df', mute='#9b968b', line='#3a3e47', acc='#82aaff', ok='#74c690', no='#e8836f', warm='#f0c36a'),
    paper=dict(bg='#faf4e6', fg='#2d2620', mute='#645b52', line='#d9ccb2', acc='#24598f', ok='#2a6a37', no='#a7352a', warm='#b8641f'))
WPS = 2.6        # words a second, when no voice gives the time
SAMPLES = 36     # frames of figure data in one move
PAUSE = 3        # seconds of countdown before a reveal
SCENE = PKG/'manim'/'film_scene.py'

## Text

The screen and the voice get plain text: the slots are filled, and the Markdown marks are removed.

In [ ]:
#| export
def plain(s) -> str:
    "Markdown marks out, for the screen and the voice."
    return re.sub(r'\*\*|__|`', '', str(s or '')).strip()

def said(s, st) -> str:
    "Text with its `{slots}` filled from the state, and no marks."
    return plain(fmt_text(s, st))

def secs_for(say:str, least:float=2.) -> float:
    "Seconds that a line needs, from its count of words."
    return max(least, len(say.split()) / WPS + .8)

A slot takes its value from the state, and the marks go.

In [ ]:
test_eq(said('First: **{p.0.id}**', dict(p=[dict(id='B')])), 'First: B')
test_eq(secs_for('one two'), 2.)

## Figures

`fig` turns one block into figure data, with the values of the state. A block that waits for an open gate does not show, and a key that an open gate hides does not show.
The film draws `axis` (with `plot` and `point`), `bars`, `grid`, `pixels`, `ranklist`, `pair`, `flow`, `readout`, `slider`, `wheel`, `toggle` and `choice`.
The voice says the captions. A 3D `scene` is not in the film yet.

In [ ]:
#| export
def _blocks(bs): return (b for _, b, _ in _walk(bs))
def _some(**kw): return {k: v for k, v in kw.items() if v is not None}

def _num(v):
    try: return float(v)
    except (TypeError, ValueError): return None

def _shown(b, opn) -> bool: return b.get('after') not in opn

def _hidden(spec, opn) -> set: return {k for g in _gates(spec.get('blocks')) if g['id'] in opn for k in g.get('hides') or ()}

The values of a list become rows of label, value and parts. A list of numbers becomes one row of cells.

In [ ]:
#| export
def _items(v) -> list:
    "Rows of `[label, value, parts]` from a derived list."
    if not isinstance(v, list): return []
    return [[str(x.get('id', i + 1)), _num(x.get('value')) or 0., x.get('parts')] if isinstance(x, dict) else [str(i + 1), _num(x) or 0., None]
            for i, x in enumerate(v)]

def _rows(v) -> list:
    "Rows of numbers from a list or a list of rows."
    if not isinstance(v, list): return []
    return [[_num(c) or 0. for c in r] for r in v] if v and isinstance(v[0], list) else [[_num(c) or 0. for c in v]]

An axis keeps its plots and its points that show.

In [ ]:
#| export
def _axis(b, st, hid, opn):
    plots = [dict(pts=[[p['x'], p['y']] for p in get(st, c['series']) or [] if _num(p.get('y')) is not None], label=c.get('label', ''))
             for c in b.get('blocks', []) if c['type'] == 'plot' and _shown(c, opn) and c['series'] not in hid]
    pts = []
    for c in b.get('blocks', []):
        if c['type'] != 'point' or not _shown(c, opn) or {c.get('x'), c.get('y'), c.get('bind')} & hid: continue
        x, y = (_num(get(st, c['x'])), _num(get(st, c['y']))) if c.get('x') else (_num(get(st, c.get('bind'))), 0.)
        if x is not None: pts.append(dict(x=x, y=y or 0., label=c.get('label', '')))
    return dict(k='axis', x=b['x'], y=b.get('y') or [-1, 1], xstep=b.get('xstep'), ystep=b.get('ystep'), xlabel=b.get('xlabel', ''),
                ylabel=b.get('ylabel', ''), plots=plots, points=pts, label=b.get('label', ''))

A slider is a knob on a track. Its value has at most two decimals, also while it moves. A wheel has its items, its outer ring, and its hand or its picked item.

In [ ]:
#| export
def _knob(b, st):
    v = get(st, b['bind'])
    return dict(k='knob', label=b.get('label') or b['bind'], value=_num(v) or 0., lo=b.get('min', 0), hi=b.get('max', 1), text=show_value(v, b.get('digits', 2)))

def _wheel(b, st):
    on = not b.get('marks_on') or bool(get(st, b['marks_on']))
    pick = b['items'].index(get(st, b['bind'])) if b.get('bind') and get(st, b['bind']) in b['items'] else None
    return dict(k='wheel', items=[plain(x) for x in b['items']], outer=[plain(x) for x in b.get('outer') or []], pick=pick,
                angle=_num(get(st, b['angle'])) if b.get('angle') else None, marks=(b.get('marks') or 0) if on else 0, label=b.get('label', ''))

`fig` sends each block type to its maker. A `row` and a `stack` hold the figures of their blocks.

In [ ]:
#| export
def fig(b, st, hid=(), opn=()):
    "One block as figure data, or None when the film does not draw it."
    t, bind = b['type'], b.get('bind')
    if not _shown(b, opn) or (bind and bind in hid): return None
    if t in ('row', 'stack'): return dict(k=t, items=xs) if (xs := [f for c in b.get('blocks', []) if (f := fig(c, st, hid, opn))]) else None
    if t == 'axis': return _axis(b, st, hid, opn)
    if t == 'bars': return dict(k='bars', label=b.get('label', ''), items=_items(get(st, bind)), max=b.get('max'), parts=b.get('parts'), digits=b.get('digits', 2))
    if t in ('grid', 'pixels'): return dict(k=t, label=said(b.get('label', ''), st), rows=_rows(get(st, bind)), **_some(cols=b.get('cols'), lo=b.get('lo'), hi=b.get('hi')))
    if t == 'ranklist': return dict(k='ranks', label=b.get('label', ''), items=[str(x) for x in get(st, bind) or []])
    if t == 'pair': return dict(k='pair', cells=[said(c, st) for c in b['cells']], to=said(b['to'], st), heads=b.get('heads'))
    if t == 'flow':
        ns = b['nodes']
        return dict(k='flow', nodes=[plain(n['text']) for n in ns], ids=[n['id'] for n in ns], layout=b.get('layout', 'row'),
                    edges=[[e['from'], e['to']] for e in b['edges']] if b.get('edges') else [[ns[i]['id'], ns[(i + 1) % len(ns)]['id']] for i in range(len(ns) - (b.get('layout') != 'cycle'))])
    if t == 'readout': return dict(k='text', text=said(b['text'], st), label=b.get('label', ''), big=True)
    if t == 'slider': return _knob(b, st)
    if t == 'wheel': return _wheel(b, st)
    if t == 'toggle': return dict(k='chip', label=b.get('label') or bind, text='on' if get(st, bind) else 'off')
    if t == 'choice': return dict(k='chip', label=b.get('label') or bind, text=show_value(get(st, bind)))
    return None

def figs(spec, st, opn=(), step=None) -> list:
    "The figures that show at state `st`, with the gates `opn` still open, and with the blocks up to `step`."
    hid, opn = _hidden(spec, opn), set(opn)
    return [f for b in spec.get('blocks', []) if step is None or (b.get('step') or 0) <= step if (f := fig(b, st, hid, opn))]

Before the gate opens, the parabola demo shows its curve and its sliders, but not the lowest point.

In [ ]:
s = gallery()['parabola-vertex']
fs = figs(s, run(s), ['move'])
test_eq([f['k'] for f in fs], ['axis', 'row'])
test_eq(fs[0]['points'], [])
test_eq(len(figs(s, run(s))[0]['points']), 1)

## Moves

A move is a change of the state that the film shows as motion. A gate prompt can name one: "b goes from 0 to 2". A caption can ask for one: "Move t to 1000".
A word names a state key, or the last word of the label of a control, or a block type ("the wheel").

In [ ]:
#| export
def _keyof(spec, word) -> str|None:
    "The state key that `word` names: the key, the last word of a control label, or the key of a block of that type."
    st = spec.get('state') or {}
    if word in st: return word
    for b in _blocks(spec.get('blocks')):
        k = b.get('bind') or b.get('angle')
        if k in st and (b['type'] == word or str(b.get('label', '')).split()[-1:] == [word]): return k
    return next((k for k in st if k.lower() == word.lower()), None)

def _num_key(spec, k): return isinstance((spec.get('state') or {}).get(k), (int, float)) and not isinstance(spec['state'][k], bool)

`gate_move` reads the prompt of a gate. `caption_moves` reads a caption.

In [ ]:
#| export
_GOES = re.compile(r'(\w+) goes from ([−-]?[\d.]+) to ([−-]?[\d.]+)')
_TRY = re.compile(r'(?:Move|Set|Turn) (?:the )?(\w+)(?: down| up| near)? to ([−-]?[\d.]+)')
def _f(v): return float(v.replace('−', '-'))

def gate_move(spec, g) -> dict:
    "The state change that a gate prompt names, as in 'b goes from 0 to 2': `{key: 2}`. Empty when it names none."
    m = _GOES.search(g.get('prompt', ''))
    k = m and _keyof(spec, m[1])
    return {k: _f(m[3])} if k and _num_key(spec, k) else {}

def caption_moves(spec, text) -> dict:
    "The state changes that a caption asks for, as in 'Move t to 1000': `{key: 1000}`."
    return {k: _f(v) for w, v in _TRY.findall(text or '') if (k := _keyof(spec, w)) and _num_key(spec, k)}

Each gallery prompt and caption gives the move that it names.

In [ ]:
g = gallery()
test_eq(gate_move(g['parabola-vertex'], next(_gates(g['parabola-vertex']['blocks']))), {'b': 2.})
test_eq(gate_move(g['softmax-temperature'], next(_gates(g['softmax-temperature']['blocks']))), {'t': 2.})
test_eq(caption_moves(g['nakshatra-pada'], 'Turn the wheel to 30°.'), {'lon': 30.})
test_eq(caption_moves(g['reciprocal-rank-fusion'], 'Move k down to 1.'), {'k': 1.})
test_eq(caption_moves(g['softmax-temperature'], 'Pick a word.'), {})

`tween` gives a state between two states. `move_frames` gives the figures at `SAMPLES` points on the way, with an ease in and out.

In [ ]:
#| export
def tween(a, b, t):
    "A value between `a` and `b` at `t` in 0..1: numbers and lists of numbers move, other values jump at the end."
    if isinstance(a, (int, float)) and isinstance(b, (int, float)): return a + (b - a) * t
    if isinstance(a, list) and isinstance(b, list) and len(a) == len(b): return [tween(x, y, t) for x, y in zip(a, b)]
    return b if t >= 1 else a

def _ease(t): return t * t * (3 - 2 * t)

def _inputs(spec, st): return {k: v for k, v in st.items() if k in (spec.get('state') or {})}

def move_frames(spec, st0, to, opn=()) -> list:
    "Figure data at `SAMPLES` points while the state goes from `st0` to `st0 | to`."
    base = {**_inputs(spec, st0), 'gate': {x: 'open' for x in opn}}
    return [figs(spec, run(spec, {**base, **{k: tween(base.get(k), v, _ease(i / (SAMPLES - 1))) for k, v in to.items()}}), opn) for i in range(SAMPLES)]

A move starts at the first state and stops at the new one.

In [ ]:
test_eq(tween(0, 2, .5), 1.)
s = gallery()['parabola-vertex']
fr = move_frames(s, run(s), {'b': 2.})
test_eq(len(fr), SAMPLES)
test_eq(fr[0][0]['points'][0]['x'], 0)
test_eq(fr[-1][0]['points'][0]['x'], -1.)

## Beats

A film is a list of beats. Each beat has a kind, a voice line `say`, a `head` on the screen, its seconds and its figure frames.

- A demo that predicts has a `title` beat, then for each gate: an `ask` with a countdown, a `move` when the prompt names one, a `reveal` with the answer and the caption, and a `move` when the caption asks for one.
- An explainer with the key `step` has a `title` beat and one `step` beat for each step. The voice says the captions of that step.

In [ ]:
#| export
def _answer(g, st) -> str:
    a = g.get('answer')
    return str(a['is']) if isinstance(a, dict) else show_value(get(st, a)) if isinstance(a, str) else ''

def _captions(spec, gid) -> list: return [b['text'] for b in _blocks(spec.get('blocks')) if b['type'] == 'caption' and b.get('after') == gid]

def _beat(kind, say, frames, head='', **kw): return dict(kind=kind, say=say, head=head, frames=frames, **{'secs': secs_for(say), **kw})

def _label(spec, k):
    "The name of key `k` for the voice: a short control label, or the key."
    l = next((b.get('label') for b in _blocks(spec.get('blocks')) if k in (b.get('bind'), b.get('angle')) and b.get('label')), '')
    return l if l and len(l.split()) <= 3 else k

def _move(spec, st, to, opn, lead, head):
    words = ', '.join(f'{_label(spec, k)} goes to {show_value(v)}' for k, v in to.items())
    return _beat('move', f'{lead}: {words}.', move_frames(spec, st, to, opn), head=head, secs=max(4., secs_for(words) + 1))

`_predict_beats` follows the demo loop, one gate after the other.

In [ ]:
#| export
def _predict_beats(spec, moves=None):
    st, gates = run(spec), list(_gates(spec.get('blocks')))
    opn = [g['id'] for g in gates]
    bs = [_beat('title', said(spec.get('ask'), st), [figs(spec, st, opn)], head=plain(spec.get('title')))]
    for g in gates:
        to = (moves or {}).get(g['id']) or gate_move(spec, g)
        opts = [plain(show_value(o)) for o in (g['options'] if isinstance(g.get('options'), list) else get(st, g['options']['from']) or [])]
        q = said(g['prompt'], st)
        bs.append(_beat('ask', f"{said(g.get('context'), st)} {q} Pause the video, and choose.".strip(), [figs(spec, st, opn)], head=q, opts=opts, secs=secs_for(q) + PAUSE))
        opn = [x for x in opn if x != g['id']]
        if to: bs.append(_move(spec, st, to, opn, 'Watch', q))
        st = run(spec, {**_inputs(spec, st), **to, 'gate': {x: 'open' for x in opn}})
        bs.append(_beat('reveal', f"The answer: {_answer(g, st)}. {' '.join(said(t, st) for t in _captions(spec, g['id']))}".strip(),
                        [figs(spec, st, opn)], head=q, opts=opts, pick=_answer(g, st)))
        if (more := caption_moves(spec, ' '.join(_captions(spec, g['id'])))):
            bs.append(_move(spec, st, more, opn, 'Now try it', 'Try it'))
            st = run(spec, {**_inputs(spec, st), **more, 'gate': {x: 'open' for x in opn}})
    return bs

`_learn_beats` builds an explainer one step at a time.

In [ ]:
#| export
def _learn_beats(spec):
    st, title = run(spec), plain(spec.get('title'))
    if 'step' not in st: return [_beat('title', said(spec.get('ask'), st), [figs(spec, st)], head=title)]
    steps = sorted({b['step'] for b in _blocks(spec.get('blocks')) if isinstance(b.get('step'), int)} | {1})
    bs = [_beat('title', said(spec.get('ask') or spec.get('title'), st), [[]], head=title)]
    for n in steps:
        s = run(spec, {**spec['state'], 'step': n})
        say = ' '.join(said(b['text'], s) for b in _blocks(spec.get('blocks')) if b['type'] == 'caption' and b.get('step') == n)
        bs.append(_beat('step', say or f'Step {n}.', [figs(spec, s, step=n)], head=title))
    return bs

## The plan

`plan` is the film data of a spec. `moves` can give the move of a gate by its id, when the prompt names none.
`timed` sets the seconds of each beat from the voice. `srt` gives the voice lines as subtitles.

In [ ]:
#| export
def plan(spec, moves:dict=None, palette:str='board', font:str='Inter') -> dict:
    "A spec as film data: one beat for each idea, each with its voice line, its seconds and its figure frames."
    bs = _learn_beats(spec) if spec.get('mode') == 'learn' else _predict_beats(spec, moves)
    return dict(id=spec['id'], title=plain(spec.get('title')), beats=bs, colors=PALETTES[palette], font=font)

def narration(p) -> list:
    "The voice line of each beat."
    return [b['say'] for b in p['beats']]

def timed(p, secs) -> dict:
    "Plan `p` with the seconds of each beat from the voice: the voice, a short rest, and the countdown of an ask."
    p = deepcopy(p)
    for b, s in zip(p['beats'], secs): b['secs'] = max(s + .6, 4. if b['kind'] == 'move' else 0) + (PAUSE if b['kind'] == 'ask' else 0)
    return p

def _ts(x): h, r = divmod(x, 3600); m, s = divmod(r, 60); return f'{int(h):02}:{int(m):02}:{int(s):02},{int(round(s % 1 * 1000)) % 1000:03}'

def srt(p) -> str:
    "The voice lines as SubRip subtitles."
    out, t = [], 0.
    for i, b in enumerate(p['beats'], 1): out.append(f"{i}\n{_ts(t)} --> {_ts(t + b['secs'])}\n{b['say']}\n"); t += b['secs']
    return '\n'.join(out)

The parabola film asks, moves b to 2, shows the answer, and then does what its caption asks: b goes to −2.

In [ ]:
p = plan(gallery()['parabola-vertex'])
test_eq([b['kind'] for b in p['beats']], ['title', 'ask', 'move', 'reveal', 'move'])
test_eq(p['beats'][3]['pick'], 'Left and down')
test_eq(p['beats'][4]['frames'][-1][0]['points'][0]['x'], 1.)
assert 'Pause the video' in p['beats'][1]['say']

The plan is JSON, so the scene gets data and no code. Each gallery demo makes a plan.

In [ ]:
for k, s in gallery().items(): json.dumps(plan(s)); assert len(plan(s)['beats']) >= 3, k

An explainer has one beat for each step, and the pair of step 2 shows only from step 2.

In [ ]:
from rupaka.spec import demo, pair, caption
ex = demo('targets', 'Make a question and its answer', mode='learn', state=dict(step=1),
          blocks=[pair(['[18]'], '47', step=1), pair(['[18, 47]'], '56', step=2), caption('One more input gives one more target.', step=2)])
p = plan(ex)
test_eq([b['kind'] for b in p['beats']], ['title', 'step', 'step'])
test_eq(len(p['beats'][1]['frames'][0]), 1)
test_eq(p['beats'][2]['say'], 'One more input gives one more target.')

`timed` uses the voice seconds, and `srt` counts them.

In [ ]:
p = timed(plan(ex), [2, 3, 4])
test_eq([b['secs'] for b in p['beats']], [2.6, 3.6, 4.6])
assert '00:00:06,200 --> 00:00:10,800' in srt(p)

## Sound

`track` makes one WAV from the WAV of each beat. Each line starts with its beat, and silence fills the rest of the beat.

In [ ]:
#| export
def _wav(frames:bytes, rate:int) -> bytes:
    out = io.BytesIO()
    with wave.open(out, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(frames)
    return out.getvalue()

def _frames(a:bytes, rate:int) -> bytes:
    with wave.open(io.BytesIO(a)) as r:
        if (r.getframerate(), r.getsampwidth(), r.getnchannels()) != (rate, 2, 1): raise ValueError(f'Each line must be mono 16-bit WAV at {rate} Hz.')
        return r.readframes(r.getnframes())

def track(wavs, secs, rate:int=None) -> bytes:
    "One WAV: the WAV of each beat at the start of its beat, and silence to the end of the beat. The rate comes from the first WAV."
    rate = rate or next((wave.open(io.BytesIO(a)).getframerate() for a in wavs if a), 24000)
    return _wav(b''.join((_frames(a, rate) if a else b'')[:int(s * rate) * 2].ljust(int(s * rate) * 2, b'\0') for a, s in zip(wavs, secs)), rate)

Two beats of one second each make two seconds of sound.

In [ ]:
one = _wav(b'\1\0' * 800, 1000)
w = track([one, None], [1, 1])
with wave.open(io.BytesIO(w)) as r: test_eq((r.getframerate(), r.getnframes()), (1000, 2000))
test_fail(lambda: track([one], [1], rate=8000), contains='mono 16-bit')

## Render

`render` writes the plan to a folder, runs manimgl on the scene there, and puts the sound in with ffmpeg. With no display, it uses `xvfb-run` when there is one.
It writes the subtitles beside the video. Install the extra with `pip install "rupaka[film]"`. manimgl needs ffmpeg, and on Linux also the pango headers.

In [ ]:
#| export
def can_film() -> bool:
    "Whether manimgl and ffmpeg are on this machine."
    return bool(shutil.which('manimgl') and shutil.which('ffmpeg'))

def _cmd(p, res) -> list:
    cmd = ['manimgl', 'film_scene.py', 'Film', '-w', '-r', res, '--video_dir', '.', '--file_name', 'film', '-c', p['colors']['bg']]
    return ['xvfb-run', '-a', '-s', '-screen 0 1920x1080x24', *cmd] if not os.environ.get('DISPLAY') and shutil.which('xvfb-run') else cmd

`render` keeps no file but the video and its subtitles.

In [ ]:
#| export
def render(p, out, wav:bytes=None, res:str='1920x1080') -> Path:
    "Draw plan `p` with manimgl into the MP4 `out`, with the sound `wav` when there is one. The subtitles go beside it as `.srt`."
    if not can_film(): raise RuntimeError('A film needs manimgl and ffmpeg: pip install "rupaka[film]".')
    out = Path(out); out.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as d:
        d = Path(d); (d/'plan.json').write_text(json.dumps(p)); shutil.copy(SCENE, d/'film_scene.py')
        r = subprocess.run(_cmd(p, res), cwd=d, env={**os.environ, 'RUPAKA_PLAN': str(d/'plan.json')}, capture_output=True, text=True)
        if not (vid := next(d.rglob('film.mp4'), None)): raise RuntimeError(f'manimgl made no video:\n{(r.stdout + r.stderr)[-3000:]}')
        if wav:
            (d/'voice.wav').write_bytes(wav)
            subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', vid, '-i', d/'voice.wav', '-c:v', 'copy', '-c:a', 'aac', '-b:a', '160k', '-shortest', out], check=True)
        else: shutil.copy(vid, out)
    out.with_suffix('.srt').write_text(srt(p))
    return out

`film` does it all for a spec: the plan, the times from the voice, the sound, and the video.

In [ ]:
#| export
def film(spec, out, wavs=None, secs=None, **kw) -> Path:
    "The video of `spec` at `out`. `wavs` and `secs` are the voice of each beat, in the order of `narration`."
    p = plan(spec, **{k: v for k, v in kw.items() if k in ('moves', 'palette', 'font')})
    if secs: p = timed(p, secs)
    return render(p, out, track(wavs, [b['secs'] for b in p['beats']]) if wavs else None, **{k: v for k, v in kw.items() if k == 'res'})

This slow test draws a short film when manimgl is here. Run it with `nbdev-test --flags slow`.

In [ ]:
#| slow
import tempfile
if can_film():
    with tempfile.TemporaryDirectory() as d:
        v = film(gallery()['parabola-vertex'], Path(d)/'p.mp4', res='640x360')
        assert v.stat().st_size > 10_000 and v.with_suffix('.srt').exists()